# 03 - Modeling and Evaluation

In the previous notebook, I used K-Means to create Steam user segments based on user-level behavioral features.

In this notebook, I use supervised learning models to check how well these discovered segments can be reproduced from the same behavioral features.  
This is not a real external prediction problem because the cluster labels are created by K-Means.  
The purpose is to make the segments more interpretable and operational.

## 1. Import libraries and project paths

I use common classification models from scikit-learn and evaluate them with accuracy, precision, recall, F1-score, and confusion matrix.

In [ ]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.cluster import KMeans

from sklearn.dummy import DummyClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier

from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.config import PROCESSED_DATA_DIR, FIGURES_DIR

## 2. Load user-level features

The modeling step starts from the processed user-level feature table.
Each row represents one Steam user.

In [ ]:
user_features_path = PROCESSED_DATA_DIR / "user_features.csv"

user_features = pd.read_csv(user_features_path)
user_features.head()

In [ ]:
user_features.shape

## 3. Create cluster labels again

The cluster labels are recreated using the same feature columns and K-Means setup from the clustering notebook.  
This keeps the modeling notebook understandable even if it is opened separately.

In [ ]:
feature_columns = [
    "total_hours",
    "avg_hours",
    "max_hours",
    "unique_games",
    "purchase_count",
    "total_interactions",
    "hours_per_game",
    "purchase_ratio"
]

X = user_features[feature_columns]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
cluster_labels = kmeans.fit_predict(X_scaled)

clustered_users = user_features.copy()
clustered_users["cluster"] = cluster_labels

clustered_users.head()

In [ ]:
clustered_users["cluster"].value_counts().sort_index()

## 4. Save clustered user table

I save the user table with cluster labels because the dashboard and later analysis can reuse it.

In [ ]:
clustered_users_path = PROCESSED_DATA_DIR / "clustered_users.csv"

clustered_users.to_csv(clustered_users_path, index=False)

clustered_users_path

## 5. Prepare X and y for classification

Here, the input variables are the user-level behavioral features.  
The target variable is the K-Means cluster label.

In [ ]:
X = clustered_users[feature_columns]
y = clustered_users["cluster"]

X.head()

In [ ]:
y.value_counts().sort_index()

## 6. Train-test split

I split the data into train and test sets.  
The split is stratified so that the cluster distribution is similar in both sets.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

## 7. Define models

I compare a simple baseline model with several classification models covered in the course.

The Dummy Classifier is important because it gives a simple baseline.  
The other models are checked against this baseline.

In [ ]:
models = {
    "Dummy Classifier": DummyClassifier(strategy="most_frequent", random_state=42),
    "Logistic Regression": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000, random_state=42))
    ]),
    "KNN": Pipeline(steps=[
        ("scaler", StandardScaler()),
        ("model", KNeighborsClassifier(n_neighbors=7))
    ]),
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42),
    "Gradient Boosting": HistGradientBoostingClassifier(max_iter=100, random_state=42)
}

## 8. Train and evaluate models

For each model, I calculate accuracy, precision, recall, and F1-score on the test set.
Since this is a multi-class problem, I use weighted averages for precision, recall, and F1-score.

In [ ]:
results = []

for model_name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    results.append({
        "Model": model_name,
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision": precision_score(y_test, y_pred, average="weighted", zero_division=0),
        "Recall": recall_score(y_test, y_pred, average="weighted", zero_division=0),
        "F1 Score": f1_score(y_test, y_pred, average="weighted", zero_division=0)
    })

results_df = pd.DataFrame(results)
results_df

In [ ]:
results_df.sort_values("F1 Score", ascending=False)

## 9. Model comparison chart

A simple bar chart makes it easier to compare the models.

In [ ]:
results_df.set_index("Model")[["Accuracy", "F1 Score"]].plot(
    kind="bar",
    figsize=(9, 4),
    title="Model Comparison",
    ylabel="Score",
    ylim=(0, 1),
    grid=True,
    rot=30
)

plt.show()

## 10. Confusion matrix for Decision Tree

I focus on the Decision Tree because it is easier to interpret than more complex models.

In [ ]:
tree_model = models["Decision Tree"]
tree_model.fit(X_train, y_train)

tree_pred = tree_model.predict(X_test)

cm = confusion_matrix(y_test, tree_pred)
cm

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))

im = ax.imshow(cm)

ax.set_title("Confusion Matrix - Decision Tree")
ax.set_xlabel("Predicted Cluster")
ax.set_ylabel("Actual Cluster")

ax.set_xticks(range(len(cm)))
ax.set_yticks(range(len(cm)))

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        ax.text(j, i, cm[i, j], ha="center", va="center")

fig.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

## 11. Classification report

The classification report gives precision, recall, and F1-score for each cluster separately.

In [ ]:
print(classification_report(y_test, tree_pred, zero_division=0))

## 12. Decision Tree feature importance

Feature importance helps show which behavioral variables are most useful for explaining the cluster labels.

In [ ]:
feature_importance = pd.DataFrame({
    "Feature": feature_columns,
    "Importance": tree_model.feature_importances_
}).sort_values("Importance", ascending=False)

feature_importance

In [ ]:
feature_importance.sort_values("Importance").plot(
    kind="barh",
    x="Feature",
    y="Importance",
    figsize=(8, 5),
    title="Decision Tree Feature Importance",
    legend=False,
    grid=True
)

plt.show()

## 13. Visualize the Decision Tree

A shallow tree is easier to read and helps explain the main decision rules behind the clusters.

In [ ]:
plt.figure(figsize=(18, 8))

plot_tree(
    tree_model,
    feature_names=feature_columns,
    class_names=[str(c) for c in sorted(y.unique())],
    filled=True,
    rounded=True,
    max_depth=3
)

plt.title("Decision Tree Explanation of User Segments")
plt.show()

## 14. Save model results

The model comparison table is saved so that it can be reused in the Streamlit dashboard.

In [ ]:
model_results_path = PROCESSED_DATA_DIR / "model_results.csv"

results_df.to_csv(model_results_path, index=False)

model_results_path

## 15. Interpretation note

The high performance of some models should be interpreted carefully.

The cluster labels are created from the same behavioral features using K-Means.  
Therefore, this modeling task does not measure prediction of an external real-world label.  
Instead, it checks whether the discovered user segments can be reproduced and explained with supervised learning models.

This is useful because it turns the cluster results into more understandable behavioral rules.

## 16. Main observations from modeling

At this stage:

- A baseline classifier was included.
- Several classification models were compared.
- Weighted precision, recall, and F1-score were used for evaluation.
- Decision Tree was used to explain the cluster labels.
- Feature importance helped identify the behavioral variables behind the segments.
- Model results were saved for the dashboard.

The next step is to add an additional recommendation-oriented analysis and then build the Streamlit dashboard.